# ML dataset v1, generate and save spin configurations

Generate the Day 25 classical Ising configuration dataset and its validation outputs. The saved spin tensor remains two-dimensional, while temperature, chain, seed and split information are stored as aligned metadata rather than spin features. The PCA and model training are not run here, it is for when i implement them in july 26th notebook.

## 1. Imports, paths and run controls

Load the established functions, find the project root and define the safe to run switches.

In [1]:
from __future__ import annotations

from pathlib import Path
from tempfile import TemporaryDirectory
from time import perf_counter
from typing import Any, Mapping, Sequence
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from IPython.display import display


def find_project_root(start: Path) -> Path:
    """Locate the repository from the notebook directory or current directory."""
    starts = [start.resolve(), Path.cwd().resolve()]
    checked: set[Path] = set()
    for first in starts:
        for candidate in (first, *first.parents):
            if candidate in checked:
                continue
            checked.add(candidate)
            if (candidate / "src" / "ising2d").is_dir() and (candidate / "configs").is_dir():
                return candidate
    raise FileNotFoundError(
        "Could not locate the project root containing src/ising2d and configs/."
    )


PROJECT_ROOT = find_project_root(Path.cwd())
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from ising2d.dataset import load_npz, make_seed_matrix, validate_temperature_grid
from ising2d.observables import (
    exact_square_lattice_critical_temperature,
    magnetisation,
    total_energy,
)
from ising2d.simulation import initialise_lattice, metropolis_sweep

RUN_SMOKE_TEST = True
RUN_FULL_DATASET = True
OVERWRITE_DAY25_CONFIGS = False
OVERWRITE_DAY25_OUTPUTS = False

CONFIG_DIR = PROJECT_ROOT / "configs"
RAW_DIR = PROJECT_ROOT / "data" / "raw" / "classical"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "classical"
FIGURE_DIR = PROJECT_ROOT / "figures" / "exploratory" / "classical" / "day25"
DAY23_CONFIG_PATH = CONFIG_DIR / "ising_day23_multiseed.yaml"
DAY23_SUMMARY_PATH = (
    PROJECT_ROOT
    / "results"
    / "processed"
    / "ising_day23_multiseed_aggregate_summary.csv"
)
DAY22_23_RAW_DIR = PROJECT_ROOT / "results" / "raw"

for directory in (CONFIG_DIR, RAW_DIR, PROCESSED_DIR, FIGURE_DIR):
    directory.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")

Project root: C:\Dev\Masters Project


## 2. Dataset settings

Reuse the Day 23 temperature grid and define the smoke and main dataset configurations. The exact square-lattice critical temperature is retained only as an infinite-system reference; it is not used as a finite-`L` class label.

**Source:** Onsager (1944), p. 117 gives the infinite-crystal transition condition, while p. 141 states that a finite crystal has analytic thermodynamic functions and no perfectly sharp transition.

In [2]:
with DAY23_CONFIG_PATH.open("r", encoding="utf-8") as handle:
    day23_config = yaml.safe_load(handle)

TEMPERATURE_GRID = validate_temperature_grid(
    day23_config["simulation"]["temperatures"]
).tolist()
EXACT_TC = exact_square_lattice_critical_temperature()


def make_day25_spec(
    *,
    name: str,
    description: str,
    lattice_size: int,
    temperatures: Sequence[float],
    burn_in_sweeps: int,
    production_sweeps: int,
    configuration_interval: int,
    n_replicates: int,
    base_seed: int,
    initial_state_schedule: Sequence[str],
    split_schedule: Sequence[str],
    filename_prefix: str,
    figure_subdirectory: str,
) -> dict[str, Any]:
    configurations_per_chain = production_sweeps // configuration_interval
    return {
        "schema_version": "1.0",
        "experiment": {
            "name": name,
            "description": description,
        },
        "model": {
            "lattice_size": lattice_size,
            "coupling": 1.0,
            "field": 0.0,
            "boltzmann_constant": 1.0,
        },
        "dataset": {
            "temperatures": [float(value) for value in temperatures],
            "burn_in_sweeps": burn_in_sweeps,
            "production_sweeps": production_sweeps,
            "configuration_interval": configuration_interval,
            "configurations_per_chain": configurations_per_chain,
            "n_replicates": n_replicates,
            "base_seed": base_seed,
            "initial_state_schedule": list(initial_state_schedule),
            "split_schedule": list(split_schedule),
        },
        "outputs": {
            "dataset_npz": f"data/raw/classical/{filename_prefix}.npz",
            "manifest_csv": f"data/processed/classical/{filename_prefix}_manifest.csv",
            "summary_csv": f"data/processed/classical/{filename_prefix}_summary.csv",
            "chain_diagnostics_csv": (
                f"data/processed/classical/{filename_prefix}_chain_diagnostics.csv"
            ),
            "figure_dir": (
                f"figures/exploratory/classical/day25/{figure_subdirectory}"
            ),
        },
    }


SMOKE_SPEC = make_day25_spec(
    name="ising_day25_dataset_smoke_test",
    description="Deterministic 24-configuration smoke dataset for the Day 25 notebook.",
    lattice_size=8,
    temperatures=[1.50, EXACT_TC, 3.50],
    burn_in_sweeps=50,
    production_sweeps=80,
    configuration_interval=20,
    n_replicates=2,
    base_seed=25200,
    initial_state_schedule=["up", "down"],
    split_schedule=["train", "test"],
    filename_prefix="ising_ml_dataset_v1_smoke",
    figure_subdirectory="smoke",
)

MAIN_SPEC = make_day25_spec(
    name="ising_day25_classical_ml_dataset_v1",
    description="Balanced L=20 raw spin-configuration dataset for later ML analysis.",
    lattice_size=20,
    temperatures=TEMPERATURE_GRID,
    burn_in_sweeps=1000,
    production_sweeps=2000,
    configuration_interval=40,
    n_replicates=6,
    base_seed=25000,
    initial_state_schedule=["up", "down", "up", "down", "up", "down"],
    split_schedule=["train", "train", "validation", "validation", "test", "test"],
    filename_prefix="ising_ml_dataset_v1",
    figure_subdirectory="main",
)

SMOKE_CONFIG_PATH = CONFIG_DIR / "ising_day25_dataset_smoke_test.yaml"
MAIN_CONFIG_PATH = CONFIG_DIR / "ising_day25_classical_ml_dataset_v1.yaml"


def write_config(path: Path, specification: Mapping[str, Any], overwrite: bool) -> None:
    """Write a configuration without silently replacing different contents."""
    if path.exists() and not overwrite:
        with path.open("r", encoding="utf-8") as handle:
            existing = yaml.safe_load(handle)
        if existing != dict(specification):
            raise FileExistsError(
                f"{path} already exists with different contents. "
                "Review or remove the old Day 25 configuration before rerunning."
            )
        return
    with path.open("w", encoding="utf-8") as handle:
        yaml.safe_dump(dict(specification), handle, sort_keys=False)


write_config(SMOKE_CONFIG_PATH, SMOKE_SPEC, OVERWRITE_DAY25_CONFIGS)
write_config(MAIN_CONFIG_PATH, MAIN_SPEC, OVERWRITE_DAY25_CONFIGS)

print(f"Smoke configurations: {3 * 2 * 4}")
print(f"Main configurations: {18 * 6 * 50}")
print(f"Exact infinite-lattice benchmark: {EXACT_TC:.15f}")

Smoke configurations: 24
Main configurations: 5400
Exact infinite-lattice benchmark: 2.269185314213022


## 3. Retain configurations from one chain

Run burn-in and production sweeps with the simulator I established in the prevoius days. Then I copy the lattice at the configured interval. This cell adds only retention and bookkeeping.

**Source:** Metropolis *et al.* (1953), p. 1088 gives the acceptance procedure. Krauth, printed p. 250, gives the local Ising Metropolis update.

In [3]:
VALID_SPLIT_NAMES = {"train", "validation", "test"}
VALID_INITIAL_STATES = {"up", "down", "random"}


def sample_configuration_chain(
    *,
    lattice_size: int,
    temperature: float,
    burn_in_sweeps: int,
    production_sweeps: int,
    configuration_interval: int,
    seed: int,
    initial_state: str,
    coupling: float = 1.0,
    field: float = 0.0,
    boltzmann_constant: float = 1.0,
) -> dict[str, Any]:
    """Run one chain and retain copied lattices at fixed production-sweep intervals."""
    if lattice_size < 2:
        raise ValueError("lattice_size must be at least two.")
    if temperature <= 0.0:
        raise ValueError("temperature must be positive.")
    if burn_in_sweeps < 0 or production_sweeps < 1:
        raise ValueError("burn-in must be non-negative and production must be positive.")
    if configuration_interval < 1 or production_sweeps % configuration_interval != 0:
        raise ValueError("configuration_interval must divide production_sweeps exactly.")
    if initial_state not in VALID_INITIAL_STATES:
        raise ValueError("initial_state must be 'up', 'down' or 'random'.")

    rng = np.random.default_rng(seed)
    lattice = initialise_lattice(lattice_size, rng, mode=initial_state)

    for _ in range(burn_in_sweeps):
        metropolis_sweep(
            lattice,
            temperature,
            rng,
            coupling=coupling,
            field=field,
            boltzmann_constant=boltzmann_constant,
        )

    n_saved = production_sweeps // configuration_interval
    configurations = np.empty((n_saved, lattice_size, lattice_size), dtype=np.int8)
    production_sweep_numbers = np.empty(n_saved, dtype=np.int32)
    energy_per_spin = np.empty(n_saved, dtype=np.float64)
    signed_magnetisation = np.empty(n_saved, dtype=np.float64)
    absolute_magnetisation = np.empty(n_saved, dtype=np.float64)
    acceptance_fraction = np.empty(n_saved, dtype=np.float64)

    accepted_fraction_sum = 0.0
    saved_index = 0
    n_spins = lattice_size**2

    for production_sweep in range(1, production_sweeps + 1):
        accepted_fraction_sum += metropolis_sweep(
            lattice,
            temperature,
            rng,
            coupling=coupling,
            field=field,
            boltzmann_constant=boltzmann_constant,
        )

        if production_sweep % configuration_interval == 0:
            configurations[saved_index] = lattice.copy()
            production_sweep_numbers[saved_index] = production_sweep
            energy_per_spin[saved_index] = (
                total_energy(lattice, coupling=coupling, field=field) / n_spins
            )
            signed_magnetisation[saved_index] = magnetisation(lattice)
            absolute_magnetisation[saved_index] = abs(signed_magnetisation[saved_index])
            acceptance_fraction[saved_index] = (
                accepted_fraction_sum / configuration_interval
            )
            accepted_fraction_sum = 0.0
            saved_index += 1

    return {
        "configurations": configurations,
        "production_sweep_numbers": production_sweep_numbers,
        "sample_indices_within_chain": np.arange(n_saved, dtype=np.int16),
        "energy_per_spin": energy_per_spin,
        "signed_magnetisation": signed_magnetisation,
        "absolute_magnetisation": absolute_magnetisation,
        "acceptance_fraction": acceptance_fraction,
        "final_lattice": lattice.copy(),
    }


## 4. Build the complete dataset

Run every temperature–replicate chain, keep the raw `±1` configurations and attach aligned provenance. Complete chains are assigned to one split so correlated states from the same trajectory are not divided between training and evaluation.

**Source:** Carrasquilla and Melko (2017), pp. 431–432 use raw Monte Carlo spin configurations as machine-learning inputs and evaluate separate test configurations. The stricter complete-chain split used here is this project's leakage-control design, not a procedure claimed by that paper.

In [4]:
def build_classical_ml_dataset(
    specification: Mapping[str, Any],
    *,
    verbose: bool = True,
) -> dict[str, Any]:
    """Build a deterministic temperature-major configuration dataset."""
    model = specification["model"]
    design = specification["dataset"]

    temperatures = validate_temperature_grid(design["temperatures"])
    lattice_size = int(model["lattice_size"])
    coupling = float(model["coupling"])
    field = float(model["field"])
    boltzmann_constant = float(model["boltzmann_constant"])
    burn_in_sweeps = int(design["burn_in_sweeps"])
    production_sweeps = int(design["production_sweeps"])
    configuration_interval = int(design["configuration_interval"])
    configurations_per_chain = int(design["configurations_per_chain"])
    n_replicates = int(design["n_replicates"])
    base_seed = int(design["base_seed"])
    initial_state_schedule = [str(value) for value in design["initial_state_schedule"]]
    split_schedule = [str(value) for value in design["split_schedule"]]

    if field != 0.0:
        raise ValueError("Classical ML dataset v1 is defined for B=0.")
    if production_sweeps // configuration_interval != configurations_per_chain:
        raise ValueError("configurations_per_chain disagrees with the production interval.")
    if len(initial_state_schedule) != n_replicates:
        raise ValueError("initial_state_schedule needs one entry per replicate.")
    if len(split_schedule) != n_replicates:
        raise ValueError("split_schedule needs one entry per replicate.")
    if set(initial_state_schedule) - VALID_INITIAL_STATES:
        raise ValueError("Unsupported initial state in initial_state_schedule.")
    if set(split_schedule) - VALID_SPLIT_NAMES:
        raise ValueError("Unsupported split name in split_schedule.")

    seed_matrix = make_seed_matrix(base_seed, n_replicates, temperatures.size)
    n_temperatures = int(temperatures.size)
    n_chains = n_temperatures * n_replicates
    n_samples = n_chains * configurations_per_chain
    exact_tc = exact_square_lattice_critical_temperature(
        coupling=coupling,
        boltzmann_constant=boltzmann_constant,
    )

    configurations = np.empty((n_samples, lattice_size, lattice_size), dtype=np.int8)
    sample_ids = np.empty(n_samples, dtype="<U24")
    sample_temperatures = np.empty(n_samples, dtype=np.float64)
    temperature_indices = np.empty(n_samples, dtype=np.int16)
    replicate_indices = np.empty(n_samples, dtype=np.int16)
    chain_indices = np.empty(n_samples, dtype=np.int32)
    chain_ids = np.empty(n_samples, dtype="<U40")
    seeds = np.empty(n_samples, dtype=np.int64)
    initial_states = np.empty(n_samples, dtype="<U6")
    production_sweep_numbers = np.empty(n_samples, dtype=np.int32)
    sample_indices_within_chain = np.empty(n_samples, dtype=np.int16)
    split_names = np.empty(n_samples, dtype="<U10")
    energy_per_spin = np.empty(n_samples, dtype=np.float64)
    signed_magnetisation = np.empty(n_samples, dtype=np.float64)
    absolute_magnetisation = np.empty(n_samples, dtype=np.float64)
    acceptance_fraction = np.empty(n_samples, dtype=np.float64)

    started = perf_counter()
    row_start = 0
    chain_index = 0

    for temperature_index, temperature in enumerate(temperatures):
        for replicate_index in range(n_replicates):
            seed = int(seed_matrix[replicate_index, temperature_index])
            initial_state = initial_state_schedule[replicate_index]
            split_name = split_schedule[replicate_index]
            chain_id = (
                f"ising2d_t{temperature_index:02d}_r{replicate_index:02d}_seed{seed}"
            )

            if verbose:
                print(
                    f"chain {chain_index + 1:03d}/{n_chains}: "
                    f"T={temperature:.9f}, replicate={replicate_index}, "
                    f"start={initial_state}, split={split_name}, seed={seed}",
                    flush=True,
                )

            chain = sample_configuration_chain(
                lattice_size=lattice_size,
                temperature=float(temperature),
                burn_in_sweeps=burn_in_sweeps,
                production_sweeps=production_sweeps,
                configuration_interval=configuration_interval,
                seed=seed,
                initial_state=initial_state,
                coupling=coupling,
                field=field,
                boltzmann_constant=boltzmann_constant,
            )

            row_stop = row_start + configurations_per_chain
            rows = slice(row_start, row_stop)
            local_indices = np.arange(configurations_per_chain, dtype=np.int16)

            configurations[rows] = chain["configurations"]
            sample_ids[rows] = [
                f"d25_t{temperature_index:02d}_r{replicate_index:02d}_s{sample_index:03d}"
                for sample_index in local_indices
            ]
            sample_temperatures[rows] = float(temperature)
            temperature_indices[rows] = temperature_index
            replicate_indices[rows] = replicate_index
            chain_indices[rows] = chain_index
            chain_ids[rows] = chain_id
            seeds[rows] = seed
            initial_states[rows] = initial_state
            production_sweep_numbers[rows] = chain["production_sweep_numbers"]
            sample_indices_within_chain[rows] = local_indices
            split_names[rows] = split_name
            energy_per_spin[rows] = chain["energy_per_spin"]
            signed_magnetisation[rows] = chain["signed_magnetisation"]
            absolute_magnetisation[rows] = chain["absolute_magnetisation"]
            acceptance_fraction[rows] = chain["acceptance_fraction"]

            row_start = row_stop
            chain_index += 1

    return {
        "schema_version": np.asarray("1.0"),
        "lattice_size": np.asarray(lattice_size, dtype=np.int32),
        "n_spins": np.asarray(lattice_size**2, dtype=np.int32),
        "coupling": np.asarray(coupling, dtype=np.float64),
        "field": np.asarray(field, dtype=np.float64),
        "boltzmann_constant": np.asarray(boltzmann_constant, dtype=np.float64),
        "burn_in_sweeps": np.asarray(burn_in_sweeps, dtype=np.int32),
        "production_sweeps": np.asarray(production_sweeps, dtype=np.int32),
        "configuration_interval": np.asarray(configuration_interval, dtype=np.int32),
        "configurations_per_chain": np.asarray(configurations_per_chain, dtype=np.int32),
        "n_replicates": np.asarray(n_replicates, dtype=np.int16),
        "n_temperatures": np.asarray(n_temperatures, dtype=np.int16),
        "n_chains": np.asarray(n_chains, dtype=np.int32),
        "n_samples": np.asarray(n_samples, dtype=np.int32),
        "row_order": np.asarray(
            "temperature_index,replicate_index,sample_index_within_chain"
        ),
        "flattening_order": np.asarray("NumPy C order (row-major)"),
        "exact_infinite_lattice_critical_temperature": np.asarray(exact_tc),
        "configurations": configurations,
        "sample_ids": sample_ids,
        "temperatures": sample_temperatures,
        "temperature_indices": temperature_indices,
        "replicate_indices": replicate_indices,
        "chain_indices": chain_indices,
        "chain_ids": chain_ids,
        "seeds": seeds,
        "initial_states": initial_states,
        "production_sweep_numbers": production_sweep_numbers,
        "sample_indices_within_chain": sample_indices_within_chain,
        "split_names": split_names,
        "energy_per_spin": energy_per_spin,
        "signed_magnetisation": signed_magnetisation,
        "absolute_magnetisation": absolute_magnetisation,
        "acceptance_fraction": acceptance_fraction,
        "seed_matrix": seed_matrix.astype(np.int64),
        "initial_state_schedule": np.asarray(initial_state_schedule, dtype="<U6"),
        "split_schedule": np.asarray(split_schedule, dtype="<U10"),
        "elapsed_seconds": float(perf_counter() - started),
    }

## 5. Manifest, summaries and correlation diagnostics

Create the row-aligned manifest and compact diagnostic tables. Consecutive Hamming distance, duplicate rate and retained-magnetisation correlation are reported as checks. It is important to note that none proves statistical independence.

**Source:** Krauth, printed p. 251, shows that local Metropolis sampling can become very slow near the transition and that widely separated updates may still yield few effectively independent samples.

In [5]:
PER_SAMPLE_FIELDS = (
    "sample_ids",
    "temperatures",
    "temperature_indices",
    "replicate_indices",
    "chain_indices",
    "chain_ids",
    "seeds",
    "initial_states",
    "production_sweep_numbers",
    "sample_indices_within_chain",
    "split_names",
    "energy_per_spin",
    "signed_magnetisation",
    "absolute_magnetisation",
    "acceptance_fraction",
)

SERIALISABLE_FIELDS = (
    "schema_version",
    "lattice_size",
    "n_spins",
    "coupling",
    "field",
    "boltzmann_constant",
    "burn_in_sweeps",
    "production_sweeps",
    "configuration_interval",
    "configurations_per_chain",
    "n_replicates",
    "n_temperatures",
    "n_chains",
    "n_samples",
    "row_order",
    "flattening_order",
    "exact_infinite_lattice_critical_temperature",
    "configurations",
    *PER_SAMPLE_FIELDS,
    "seed_matrix",
    "initial_state_schedule",
    "split_schedule",
)


def serialisable_payload(dataset: Mapping[str, Any]) -> dict[str, np.ndarray]:
    payload = {name: np.asarray(dataset[name]) for name in SERIALISABLE_FIELDS}
    object_fields = [name for name, value in payload.items() if value.dtype == object]
    if object_fields:
        raise ValueError(f"Object arrays are not allowed: {object_fields}")
    return payload


def load_previous_day_seeds(raw_directory: Path) -> np.ndarray:
    """Collect recorded Day 22 and Day 23 seeds for the non-overlap check."""
    arrays: list[np.ndarray] = []
    for path in sorted(raw_directory.glob("ising_day2[23]*.npz")):
        with np.load(path, allow_pickle=False) as saved:
            if "seeds" in saved.files:
                arrays.append(np.asarray(saved["seeds"], dtype=np.int64).ravel())
    if not arrays:
        return np.empty(0, dtype=np.int64)
    return np.unique(np.concatenate(arrays))


def manifest_from_dataset(dataset: Mapping[str, Any]) -> pd.DataFrame:
    return pd.DataFrame(
        {
            "row_index": np.arange(int(dataset["n_samples"]), dtype=np.int32),
            "sample_id": dataset["sample_ids"],
            "temperature": dataset["temperatures"],
            "temperature_index": dataset["temperature_indices"],
            "replicate_index": dataset["replicate_indices"],
            "chain_index": dataset["chain_indices"],
            "chain_id": dataset["chain_ids"],
            "seed": dataset["seeds"],
            "initial_state": dataset["initial_states"],
            "production_sweep_number": dataset["production_sweep_numbers"],
            "sample_index_within_chain": dataset["sample_indices_within_chain"],
            "split_name": dataset["split_names"],
            "energy_per_spin": dataset["energy_per_spin"],
            "signed_magnetisation": dataset["signed_magnetisation"],
            "absolute_magnetisation": dataset["absolute_magnetisation"],
            "acceptance_fraction": dataset["acceptance_fraction"],
        }
    )


def make_summary_table(manifest: pd.DataFrame) -> pd.DataFrame:
    return (
        manifest.groupby(
            ["temperature_index", "temperature", "split_name", "initial_state"],
            as_index=False,
        )
        .agg(
            configuration_count=("sample_id", "size"),
            chain_count=("chain_id", "nunique"),
            mean_energy_per_spin=("energy_per_spin", "mean"),
            mean_signed_magnetisation=("signed_magnetisation", "mean"),
            mean_absolute_magnetisation=("absolute_magnetisation", "mean"),
            mean_acceptance_fraction=("acceptance_fraction", "mean"),
        )
        .sort_values(["temperature_index", "split_name", "initial_state"])
        .reset_index(drop=True)
    )

def lag_one_correlation(values: np.ndarray) -> float:
    values = np.asarray(values, dtype=float)
    if values.size < 3 or np.std(values[:-1]) == 0.0 or np.std(values[1:]) == 0.0:
        return float("nan")
    return float(np.corrcoef(values[:-1], values[1:])[0, 1])


def chain_spacing_diagnostics(dataset: Mapping[str, Any]) -> pd.DataFrame:
    configurations = np.asarray(dataset["configurations"], dtype=np.int8)
    manifest = manifest_from_dataset(dataset)
    rows: list[dict[str, Any]] = []

    for chain_index, chain_rows in manifest.groupby("chain_index", sort=True):
        chain_rows = chain_rows.sort_values("sample_index_within_chain")
        indices = chain_rows["row_index"].to_numpy(dtype=int)
        flattened = configurations[indices].reshape(indices.size, -1)
        consecutive_hamming = np.mean(flattened[1:] != flattened[:-1], axis=1)
        consecutive_duplicates = np.all(flattened[1:] == flattened[:-1], axis=1)
        unique_fraction = np.unique(flattened, axis=0).shape[0] / flattened.shape[0]

        first = chain_rows.iloc[0]
        rows.append(
            {
                "chain_index": int(chain_index),
                "chain_id": first["chain_id"],
                "temperature_index": int(first["temperature_index"]),
                "temperature": float(first["temperature"]),
                "replicate_index": int(first["replicate_index"]),
                "seed": int(first["seed"]),
                "initial_state": first["initial_state"],
                "split_name": first["split_name"],
                "configuration_count": int(indices.size),
                "mean_consecutive_hamming_fraction": float(np.mean(consecutive_hamming)),
                "minimum_consecutive_hamming_fraction": float(np.min(consecutive_hamming)),
                "consecutive_duplicate_rate": float(np.mean(consecutive_duplicates)),
                "all_rows_duplicate_rate": float(1.0 - unique_fraction),
                "signed_magnetisation_lag1": lag_one_correlation(
                    chain_rows["signed_magnetisation"].to_numpy()
                ),
                "absolute_magnetisation_lag1": lag_one_correlation(
                    chain_rows["absolute_magnetisation"].to_numpy()
                ),
            }
        )

    return pd.DataFrame(rows)


## 6. Dataset validation

Check spin integrity, provenance alignment, seed and chain separation, physical observables, low-temperature sector balance and deterministic row ordering.

In [6]:
def validate_dataset(
    dataset: Mapping[str, Any],
    specification: Mapping[str, Any],
    previous_seeds: np.ndarray,
) -> dict[str, Any]:
    payload = serialisable_payload(dataset)
    configurations = payload["configurations"]
    n_samples = int(payload["n_samples"])
    lattice_size = int(payload["lattice_size"])
    n_temperatures = int(payload["n_temperatures"])
    n_replicates = int(payload["n_replicates"])
    configurations_per_chain = int(payload["configurations_per_chain"])
    interval = int(payload["configuration_interval"])

    expected_shape = (n_samples, lattice_size, lattice_size)
    if configurations.shape != expected_shape:
        raise ValueError(f"Expected configurations.shape={expected_shape}, got {configurations.shape}.")
    if configurations.dtype != np.int8:
        raise ValueError("configurations must have dtype int8.")
    if not np.all(np.isin(configurations, (-1, 1))):
        raise ValueError("configurations must contain only -1 and +1.")

    for name in PER_SAMPLE_FIELDS:
        values = payload[name]
        if values.ndim != 1 or values.shape[0] != n_samples:
            raise ValueError(f"{name} is not aligned with configurations.")
    if np.unique(payload["sample_ids"]).size != n_samples:
        raise ValueError("sample_ids are not unique.")

    expected_samples = n_temperatures * n_replicates * configurations_per_chain
    if n_samples != expected_samples:
        raise ValueError("n_samples disagrees with the configured design.")

    chain_indices = payload["chain_indices"].astype(int)
    chain_ids = payload["chain_ids"].astype(str)
    seeds = payload["seeds"].astype(np.int64)
    split_names = payload["split_names"].astype(str)
    temperature_indices = payload["temperature_indices"].astype(int)
    sample_indices = payload["sample_indices_within_chain"].astype(int)
    sweep_numbers = payload["production_sweep_numbers"].astype(int)

    for chain_index in np.unique(chain_indices):
        mask = chain_indices == chain_index
        order = np.argsort(sample_indices[mask])
        if np.sum(mask) != configurations_per_chain:
            raise ValueError(f"Chain {chain_index} has the wrong number of configurations.")
        for values, name in (
            (chain_ids[mask], "chain_id"),
            (seeds[mask], "seed"),
            (split_names[mask], "split_name"),
            (temperature_indices[mask], "temperature_index"),
        ):
            if np.unique(values).size != 1:
                raise ValueError(f"{name} changes inside chain {chain_index}.")
        if not np.array_equal(
            sample_indices[mask][order],
            np.arange(configurations_per_chain),
        ):
            raise ValueError(f"Chain {chain_index} has incomplete sample indices.")
        expected_sweeps = interval * np.arange(1, configurations_per_chain + 1)
        if not np.array_equal(sweep_numbers[mask][order], expected_sweeps):
            raise ValueError(f"Chain {chain_index} has incorrect retained sweep numbers.")

    chain_split_pairs = pd.DataFrame(
        {"chain_id": chain_ids, "seed": seeds, "split_name": split_names}
    ).drop_duplicates()
    if chain_split_pairs.groupby("chain_id")["split_name"].nunique().max() != 1:
        raise ValueError("A chain appears in more than one split.")
    if chain_split_pairs.groupby("seed")["split_name"].nunique().max() != 1:
        raise ValueError("A seed appears in more than one split.")
    if chain_split_pairs["seed"].nunique() != int(payload["n_chains"]):
        raise ValueError("Each chain must have a unique seed.")

    for split_name in np.unique(split_names):
        represented = np.unique(temperature_indices[split_names == split_name])
        if not np.array_equal(represented, np.arange(n_temperatures)):
            raise ValueError(f"Split {split_name} does not contain the complete temperature grid.")

    if previous_seeds.size and np.intersect1d(np.unique(seeds), previous_seeds).size:
        raise ValueError("Day 25 seeds overlap recorded Day 22 or Day 23 seeds.")

    flattened = configurations.reshape(n_samples, lattice_size**2, order="C")
    if not np.array_equal(
        flattened.reshape(configurations.shape, order="C"),
        configurations,
    ):
        raise ValueError("C-order flattening and reshaping did not round-trip.")

    recomputed_energy = np.asarray(
        [
            total_energy(
                configuration,
                coupling=float(payload["coupling"]),
                field=float(payload["field"]),
            )
            / configuration.size
            for configuration in configurations
        ]
    )
    recomputed_magnetisation = np.asarray(
        [magnetisation(configuration) for configuration in configurations]
    )
    if not np.allclose(recomputed_energy, payload["energy_per_spin"], atol=1e-12, rtol=0.0):
        raise ValueError("Stored energies disagree with recomputation from configurations.")
    if not np.allclose(
        recomputed_magnetisation,
        payload["signed_magnetisation"],
        atol=1e-12,
        rtol=0.0,
    ):
        raise ValueError("Stored signed magnetisations disagree with recomputation.")
    if not np.allclose(
        np.abs(recomputed_magnetisation),
        payload["absolute_magnetisation"],
        atol=1e-12,
        rtol=0.0,
    ):
        raise ValueError("Stored absolute magnetisations disagree with recomputation.")

    test_configuration = configurations[0]
    if not np.isclose(
        total_energy(test_configuration, field=0.0),
        total_energy(-test_configuration, field=0.0),
    ):
        raise ValueError("Global spin flip did not preserve energy at B=0.")
    if not np.isclose(magnetisation(-test_configuration), -magnetisation(test_configuration)):
        raise ValueError("Global spin flip did not reverse magnetisation.")

    temperatures = payload["temperatures"].astype(float)
    signed_m = payload["signed_magnetisation"].astype(float)
    absolute_m = payload["absolute_magnetisation"].astype(float)
    energy = payload["energy_per_spin"].astype(float)
    low_temperature = float(np.min(temperatures))
    high_temperature = float(np.max(temperatures))
    low_mask = temperatures == low_temperature
    high_mask = temperatures == high_temperature
    low_signs = set(np.sign(signed_m[low_mask]).astype(int))
    if not {-1, 1}.issubset(low_signs):
        raise ValueError("Both low-temperature magnetisation sectors are not represented.")

    for split_name in np.unique(split_names):
        split_low_signs = set(
            np.sign(signed_m[low_mask & (split_names == split_name)]).astype(int)
        )
        configured_starts = set(
            payload["initial_states"][low_mask & (split_names == split_name)].astype(str)
        )
        if {"up", "down"}.issubset(configured_starts) and not {-1, 1}.issubset(split_low_signs):
            raise ValueError(
                f"Split {split_name} does not retain both low-temperature sectors."
            )

    diagnostics = chain_spacing_diagnostics(dataset)
    return {
        "all_integrity_checks_passed": True,
        "configuration_shape": list(configurations.shape),
        "configuration_dtype": str(configurations.dtype),
        "n_samples": n_samples,
        "n_chains": int(payload["n_chains"]),
        "unique_seed_count": int(np.unique(seeds).size),
        "low_temperature_mean_absolute_magnetisation": float(np.mean(absolute_m[low_mask])),
        "high_temperature_mean_absolute_magnetisation": float(np.mean(absolute_m[high_mask])),
        "low_temperature_mean_energy_per_spin": float(np.mean(energy[low_mask])),
        "high_temperature_mean_energy_per_spin": float(np.mean(energy[high_mask])),
        "both_low_temperature_sectors": True,
        "mean_consecutive_hamming_fraction": float(
            diagnostics["mean_consecutive_hamming_fraction"].mean()
        ),
        "mean_consecutive_duplicate_rate": float(
            diagnostics["consecutive_duplicate_rate"].mean()
        ),
    }

## 7. Save and reload the required outputs

Save one NPZ dataset, one row manifest, one compact summary and one chain-diagnostic table. A separate dataset-card Markdown file and separate metadata JSON are deliberately omitted because the YAML configuration, NPZ arrays and manifest already contain the required information.

In [7]:
def resolve_outputs(
    specification: Mapping[str, Any],
    *,
    output_root: Path = PROJECT_ROOT,
) -> dict[str, Path]:
    return {
        key: output_root / relative_path
        for key, relative_path in specification["outputs"].items()
    }


def require_output_paths_available(
    specification: Mapping[str, Any],
    *,
    overwrite: bool,
    output_root: Path = PROJECT_ROOT,
) -> dict[str, Path]:
    paths = resolve_outputs(specification, output_root=output_root)
    files = [path for key, path in paths.items() if key != "figure_dir"]
    if not overwrite:
        existing = [path for path in files if path.exists()]
        if paths["figure_dir"].exists():
            existing.extend(sorted(paths["figure_dir"].glob("*.png")))
        if existing:
            joined = "\n".join(f"  - {path}" for path in existing)
            raise FileExistsError(
                "Refusing to replace existing Day 25 outputs:\n" + joined
            )
    return paths


def save_dataset(
    specification: Mapping[str, Any],
    dataset: Mapping[str, Any],
    *,
    overwrite: bool,
    output_root: Path = PROJECT_ROOT,
) -> dict[str, Any]:
    paths = require_output_paths_available(
        specification,
        overwrite=overwrite,
        output_root=output_root,
    )
    for key, path in paths.items():
        if key == "figure_dir":
            path.mkdir(parents=True, exist_ok=True)
        else:
            path.parent.mkdir(parents=True, exist_ok=True)

    payload = serialisable_payload(dataset)
    manifest = manifest_from_dataset(dataset)
    summary = make_summary_table(manifest)
    diagnostics = chain_spacing_diagnostics(dataset)

    np.savez_compressed(paths["dataset_npz"], **payload)
    manifest.to_csv(paths["manifest_csv"], index=False)
    summary.to_csv(paths["summary_csv"], index=False)
    diagnostics.to_csv(paths["chain_diagnostics_csv"], index=False)

    reloaded = load_npz(paths["dataset_npz"])
    if set(reloaded) != set(payload):
        raise ValueError("Reloaded NPZ fields differ from the saved payload.")
    for name in payload:
        if not np.array_equal(reloaded[name], payload[name]):
            raise ValueError(f"Reloaded field {name} differs from the in-memory array.")

    saved_manifest = pd.read_csv(paths["manifest_csv"])
    if not np.array_equal(
        saved_manifest["sample_id"].astype(str).to_numpy(),
        np.asarray(dataset["sample_ids"]).astype(str),
    ):
        raise ValueError("Manifest rows are not aligned with NPZ rows.")

    return {
        "paths": paths,
        "manifest": manifest,
        "summary": summary,
        "diagnostics": diagnostics,
        "reloaded": reloaded,
    }

## 8. Validation figures

Create the representative-configuration montage, split counts, dataset-derived physics check and spacing diagnostics.

In [8]:
def create_validation_figures(
    specification: Mapping[str, Any],
    dataset: Mapping[str, Any],
    diagnostics: pd.DataFrame,
    *,
    overwrite: bool,
    output_root: Path = PROJECT_ROOT,
) -> tuple[dict[str, Path], pd.DataFrame]:
    paths = resolve_outputs(specification, output_root=output_root)
    figure_dir = paths["figure_dir"]
    figure_dir.mkdir(parents=True, exist_ok=True)
    prefix = specification["experiment"]["name"]
    figure_paths = {
        "montage": figure_dir / f"{prefix}_configuration_montage.png",
        "counts": figure_dir / f"{prefix}_counts_by_temperature_and_split.png",
        "physics": figure_dir / f"{prefix}_dataset_physics_check.png",
        "spacing": figure_dir / f"{prefix}_configuration_spacing_diagnostic.png",
    }
    if not overwrite:
        existing = [path for path in figure_paths.values() if path.exists()]
        if existing:
            raise FileExistsError(f"Refusing to overwrite figures: {existing}")

    manifest = manifest_from_dataset(dataset)
    configurations = np.asarray(dataset["configurations"])
    exact_tc = float(dataset["exact_infinite_lattice_critical_temperature"])
    low_temperature = float(manifest["temperature"].min())
    high_temperature = float(manifest["temperature"].max())
    near_temperature = float(
        manifest.loc[(manifest["temperature"] - exact_tc).abs().idxmin(), "temperature"]
    )

    def row_for(mask: pd.Series, *, chain_number: int = 0) -> pd.Series:
        candidates = manifest.loc[mask]
        chain_id = candidates["chain_id"].drop_duplicates().iloc[chain_number]
        return candidates.loc[candidates["chain_id"] == chain_id].iloc[-1]

    montage_rows = [
        manifest.loc[
            (manifest["temperature"] == low_temperature)
            & (manifest["signed_magnetisation"] > 0)
        ].sort_values("signed_magnetisation").iloc[-1],
        manifest.loc[
            (manifest["temperature"] == low_temperature)
            & (manifest["signed_magnetisation"] < 0)
        ].sort_values("signed_magnetisation").iloc[0],
        row_for(manifest["temperature"] == near_temperature, chain_number=0),
        row_for(manifest["temperature"] == near_temperature, chain_number=1),
        row_for(manifest["temperature"] == high_temperature, chain_number=0),
        row_for(manifest["temperature"] == high_temperature, chain_number=1),
    ]

    figure, axes = plt.subplots(2, 3, figsize=(12, 8))
    for axis, row in zip(axes.ravel(), montage_rows, strict=True):
        index = int(row["row_index"])
        axis.imshow(
            configurations[index],
            cmap="binary",
            vmin=-1,
            vmax=1,
            interpolation="nearest",
        )
        axis.set_title(
            f"T={row['temperature']:.6f}; chain={int(row['chain_index'])}; seed={int(row['seed'])}\n"
            f"start={row['initial_state']}; m={row['signed_magnetisation']:+.3f}; "
            f"sweep={int(row['production_sweep_number'])}",
            fontsize=8,
        )
        axis.set_xticks([])
        axis.set_yticks([])
    figure.suptitle("Classical Ising configuration validation montage")
    figure.tight_layout()
    figure.savefig(figure_paths["montage"], dpi=250, bbox_inches="tight")
    plt.close(figure)

    counts = (
        manifest.groupby(["temperature", "split_name"], as_index=False)
        .size()
        .rename(columns={"size": "configuration_count"})
    )
    figure, axis = plt.subplots(figsize=(9, 5))
    for split_name, rows in counts.groupby("split_name"):
        axis.plot(
            rows["temperature"],
            rows["configuration_count"],
            marker="o",
            label=split_name,
        )
    axis.set_xlabel(r"Temperature, $k_B T/J$")
    axis.set_ylabel("Saved configurations")
    axis.set_title("Dataset count by temperature and complete-chain split")
    axis.grid(True, alpha=0.3)
    axis.legend()
    figure.tight_layout()
    figure.savefig(figure_paths["counts"], dpi=250, bbox_inches="tight")
    plt.close(figure)

    day25_physics = (
        manifest.groupby("temperature", as_index=False)
        .agg(
            day25_energy_per_spin=("energy_per_spin", "mean"),
            day25_absolute_magnetisation=("absolute_magnetisation", "mean"),
        )
        .sort_values("temperature")
    )
    comparison = day25_physics.copy()
    if DAY23_SUMMARY_PATH.exists():
        day23 = pd.read_csv(DAY23_SUMMARY_PATH)[
            [
                "temperature",
                "mean_energy_per_spin_mean",
                "mean_absolute_magnetisation_mean",
            ]
        ].rename(
            columns={
                "mean_energy_per_spin_mean": "day23_energy_per_spin",
                "mean_absolute_magnetisation_mean": "day23_absolute_magnetisation",
            }
        )
        comparison = day25_physics.merge(day23, on="temperature", how="left")
        comparison["energy_difference_day25_minus_day23"] = (
            comparison["day25_energy_per_spin"] - comparison["day23_energy_per_spin"]
        )
        comparison["absolute_magnetisation_difference_day25_minus_day23"] = (
            comparison["day25_absolute_magnetisation"]
            - comparison["day23_absolute_magnetisation"]
        )

    figure, axes = plt.subplots(1, 2, figsize=(12, 4.8))
    axes[0].plot(
        comparison["temperature"],
        comparison["day25_energy_per_spin"],
        marker="o",
        label="Day 25 saved configurations",
    )
    axes[1].plot(
        comparison["temperature"],
        comparison["day25_absolute_magnetisation"],
        marker="o",
        label="Day 25 saved configurations",
    )
    if "day23_energy_per_spin" in comparison:
        axes[0].plot(
            comparison["temperature"],
            comparison["day23_energy_per_spin"],
            marker="x",
            linestyle="--",
            label="Day 23 baseline",
        )
        axes[1].plot(
            comparison["temperature"],
            comparison["day23_absolute_magnetisation"],
            marker="x",
            linestyle="--",
            label="Day 23 baseline",
        )
    for axis in axes:
        axis.axvline(exact_tc, linestyle=":", linewidth=1.0, label="exact infinite-L benchmark")
        axis.set_xlabel(r"Temperature, $k_B T/J$")
        axis.grid(True, alpha=0.3)
        axis.legend(fontsize=8)
    axes[0].set_ylabel("Mean energy per spin")
    axes[0].set_title("Energy from saved configurations")
    axes[1].set_ylabel("Mean absolute magnetisation")
    axes[1].set_title("Magnetic order from saved configurations")
    figure.tight_layout()
    figure.savefig(figure_paths["physics"], dpi=250, bbox_inches="tight")
    plt.close(figure)

    spacing = (
        diagnostics.groupby("temperature", as_index=False)
        .agg(
            mean_consecutive_hamming_fraction=(
                "mean_consecutive_hamming_fraction",
                "mean",
            ),
            mean_consecutive_duplicate_rate=("consecutive_duplicate_rate", "mean"),
            mean_signed_magnetisation_lag1=("signed_magnetisation_lag1", "mean"),
        )
        .sort_values("temperature")
    )
    figure, axes = plt.subplots(1, 3, figsize=(15, 4.6))
    axes[0].plot(
        spacing["temperature"],
        spacing["mean_consecutive_hamming_fraction"],
        marker="o",
    )
    axes[1].plot(
        spacing["temperature"],
        spacing["mean_consecutive_duplicate_rate"],
        marker="o",
    )
    axes[2].plot(
        spacing["temperature"],
        spacing["mean_signed_magnetisation_lag1"],
        marker="o",
    )
    axes[0].set_ylabel("Mean consecutive Hamming fraction")
    axes[1].set_ylabel("Consecutive exact-duplicate proportion")
    axes[2].set_ylabel("Retained signed-m lag-one correlation")
    for axis in axes:
        axis.axvline(exact_tc, linestyle=":", linewidth=1.0)
        axis.set_xlabel(r"Temperature, $k_B T/J$")
        axis.grid(True, alpha=0.3)
    figure.suptitle(
        "Configuration-spacing diagnostics (Hamming distance is not proof of independence)"
    )
    figure.tight_layout()
    figure.savefig(figure_paths["spacing"], dpi=250, bbox_inches="tight")
    plt.close(figure)

    return figure_paths, comparison

## 9. Fast tests before data generation

Exercise the new notebook functions on tiny deterministic chains before running the smoke or main dataset.

In [9]:
tiny_chain_a = sample_configuration_chain(
    lattice_size=4,
    temperature=2.0,
    burn_in_sweeps=2,
    production_sweeps=6,
    configuration_interval=2,
    seed=25,
    initial_state="up",
)
tiny_chain_b = sample_configuration_chain(
    lattice_size=4,
    temperature=2.0,
    burn_in_sweeps=2,
    production_sweeps=6,
    configuration_interval=2,
    seed=25,
    initial_state="up",
)

assert tiny_chain_a["configurations"].shape == (3, 4, 4)
assert tiny_chain_a["configurations"].dtype == np.int8
assert np.all(np.isin(tiny_chain_a["configurations"], (-1, 1)))
assert np.array_equal(tiny_chain_a["configurations"], tiny_chain_b["configurations"])
assert not np.shares_memory(
    tiny_chain_a["configurations"][0],
    tiny_chain_a["final_lattice"],
)

saved_copy = tiny_chain_a["configurations"].copy()
tiny_chain_a["final_lattice"][0, 0] *= -1
assert np.array_equal(tiny_chain_a["configurations"], saved_copy)

flattened = saved_copy.reshape(saved_copy.shape[0], -1, order="C")
assert np.array_equal(flattened.reshape(saved_copy.shape, order="C"), saved_copy)
assert np.isclose(total_energy(saved_copy[0]), total_energy(-saved_copy[0]))
assert np.isclose(magnetisation(-saved_copy[0]), -magnetisation(saved_copy[0]))

TINY_SPEC = make_day25_spec(
    name="day25_notebook_function_test",
    description="In-memory function test only.",
    lattice_size=4,
    temperatures=[1.5, 3.5],
    burn_in_sweeps=2,
    production_sweeps=4,
    configuration_interval=2,
    n_replicates=2,
    base_seed=25900,
    initial_state_schedule=["up", "down"],
    split_schedule=["train", "test"],
    filename_prefix="unused_tiny_test",
    figure_subdirectory="unused_tiny_test",
)
tiny_dataset = build_classical_ml_dataset(TINY_SPEC, verbose=False)
assert tiny_dataset["configurations"].shape == (8, 4, 4)
assert np.unique(tiny_dataset["chain_ids"]).size == 4
assert np.unique(tiny_dataset["seeds"]).size == 4
assert not np.any(tiny_dataset["split_names"][tiny_dataset["chain_indices"] == 0] != "train")

print("Fast Day 25 notebook tests passed.")

Fast Day 25 notebook tests passed.


## 10. Deterministic smoke dataset

Run the small dataset twice and compare every saved array. Its temporary files and figures are deleted automatically after save/reload validation; only the small smoke YAML is retained as a reusable regression recipe.

In [10]:
PREVIOUS_DAY_SEEDS = load_previous_day_seeds(DAY22_23_RAW_DIR)

if RUN_SMOKE_TEST:
    smoke_dataset_first = build_classical_ml_dataset(SMOKE_SPEC, verbose=False)
    smoke_dataset_second = build_classical_ml_dataset(SMOKE_SPEC, verbose=False)

    first_payload = serialisable_payload(smoke_dataset_first)
    second_payload = serialisable_payload(smoke_dataset_second)
    for field_name in first_payload:
        assert np.array_equal(
            first_payload[field_name],
            second_payload[field_name],
        ), field_name

    smoke_validation = validate_dataset(
        smoke_dataset_first,
        SMOKE_SPEC,
        PREVIOUS_DAY_SEEDS,
    )

    # The smoke output is written to a temporary directory, checked, and removed.
    with TemporaryDirectory() as temporary_directory:
        temporary_root = Path(temporary_directory)
        smoke_saved = save_dataset(
            SMOKE_SPEC,
            smoke_dataset_first,
            overwrite=False,
            output_root=temporary_root,
        )
        create_validation_figures(
            SMOKE_SPEC,
            smoke_dataset_first,
            smoke_saved["diagnostics"],
            overwrite=False,
            output_root=temporary_root,
        )

    print("Smoke dataset reproduced exactly in two independent runs.")
    print(f"Shape: {smoke_dataset_first['configurations'].shape}")
    print(smoke_validation)
    display(smoke_saved["manifest"].head(8))
    display(smoke_saved["diagnostics"])
else:
    print("Smoke run skipped because RUN_SMOKE_TEST=False.")

Smoke dataset reproduced exactly in two independent runs.
Shape: (24, 8, 8)
{'all_integrity_checks_passed': True, 'configuration_shape': [24, 8, 8], 'configuration_dtype': 'int8', 'n_samples': 24, 'n_chains': 6, 'unique_seed_count': 6, 'low_temperature_mean_absolute_magnetisation': 0.98828125, 'high_temperature_mean_absolute_magnetisation': 0.1640625, 'low_temperature_mean_energy_per_spin': -1.953125, 'high_temperature_mean_energy_per_spin': -0.734375, 'both_low_temperature_sectors': True, 'mean_consecutive_hamming_fraction': 0.21701388888888887, 'mean_consecutive_duplicate_rate': 0.05555555555555555}


,row_index,sample_id,temperature,temperature_index,replicate_index,chain_index,chain_id,seed,initial_state,production_sweep_number,sample_index_within_chain,split_name,energy_per_spin,signed_magnetisation,absolute_magnetisation,acceptance_fraction
0,0,d25_t00_r00_s000,1.5,0,0,0,ising2d_t00_r00_seed25200,25200,up,20,0,train,-2.000,1.00000,1.00000,0.001563
1,1,d25_t00_r00_s001,1.5,0,0,0,ising2d_t00_r00_seed25200,25200,up,40,1,train,-2.000,1.00000,1.00000,0.012500
2,2,d25_t00_r00_s002,1.5,0,0,0,ising2d_t00_r00_seed25200,25200,up,60,2,train,-1.875,0.96875,0.96875,0.021094
3,3,d25_t00_r00_s003,1.5,0,0,0,ising2d_t00_r00_seed25200,25200,up,80,3,train,-2.000,1.00000,1.00000,0.014844
4,4,d25_t00_r01_s000,1.5,0,1,1,ising2d_t00_r01_seed25203,25203,down,20,0,test,-2.000,-1.00000,1.00000,0.009375
5,5,d25_t00_r01_s001,1.5,0,1,1,ising2d_t00_r01_seed25203,25203,down,40,1,test,-1.875,-0.96875,0.96875,0.017969
6,6,d25_t00_r01_s002,1.5,0,1,1,ising2d_t00_r01_seed25203,25203,down,60,2,test,-2.000,-1.00000,1.00000,0.003906
7,7,d25_t00_r01_s003,1.5,0,1,1,ising2d_t00_r01_seed25203,25203,down,80,3,test,-1.875,-0.96875,0.96875,0.016406


,chain_index,chain_id,temperature_index,temperature,replicate_index,seed,initial_state,split_name,configuration_count,mean_consecutive_hamming_fraction,minimum_consecutive_hamming_fraction,consecutive_duplicate_rate,all_rows_duplicate_rate,signed_magnetisation_lag1,absolute_magnetisation_lag1
0,0,ising2d_t00_r00_seed25200,0,1.500000,0,25200,up,train,4,0.010417,0.000000,0.333333,0.50,-0.500000,-0.500000
1,1,ising2d_t00_r01_seed25203,0,1.500000,1,25203,down,test,4,0.015625,0.015625,0.000000,0.25,-1.000000,-1.000000
2,2,ising2d_t01_r00_seed25201,1,2.269185,0,25201,up,train,4,0.187500,0.109375,0.000000,0.00,0.628619,0.628619
3,3,ising2d_t01_r01_seed25204,1,2.269185,1,25204,down,test,4,0.119792,0.109375,0.000000,0.00,-0.866025,-0.866025
4,4,ising2d_t02_r00_seed25202,2,3.500000,0,25202,up,train,4,0.531250,0.515625,0.000000,0.00,-0.743487,-0.182917
5,5,ising2d_t02_r01_seed25205,2,3.500000,1,25205,down,test,4,0.437500,0.375000,0.000000,0.00,-0.363427,-0.685680


## 11. Generate and save Classical ML dataset v1

Generate the full 5,400-configuration dataset, save the persistent outputs and display compact validation summaries.

In [11]:
if RUN_FULL_DATASET:
    require_output_paths_available(
        MAIN_SPEC,
        overwrite=OVERWRITE_DAY25_OUTPUTS,
    )

    main_dataset = build_classical_ml_dataset(MAIN_SPEC, verbose=True)
    main_validation = validate_dataset(
        main_dataset,
        MAIN_SPEC,
        PREVIOUS_DAY_SEEDS,
    )
    main_saved = save_dataset(
        MAIN_SPEC,
        main_dataset,
        overwrite=OVERWRITE_DAY25_OUTPUTS,
    )
    main_figures, day23_comparison = create_validation_figures(
        MAIN_SPEC,
        main_dataset,
        main_saved["diagnostics"],
        overwrite=OVERWRITE_DAY25_OUTPUTS,
    )

    dataset_size_mib = main_saved["paths"]["dataset_npz"].stat().st_size / 1024**2
    counts = (
        main_saved["manifest"]
        .groupby(["temperature", "split_name"], as_index=False)
        .size()
        .rename(columns={"size": "configuration_count"})
    )
    spacing_by_temperature = (
        main_saved["diagnostics"]
        .groupby("temperature", as_index=False)
        .agg(
            mean_consecutive_hamming_fraction=(
                "mean_consecutive_hamming_fraction",
                "mean",
            ),
            mean_consecutive_duplicate_rate=("consecutive_duplicate_rate", "mean"),
            mean_signed_magnetisation_lag1=("signed_magnetisation_lag1", "mean"),
        )
    )

    print(f"Saved dataset shape: {main_dataset['configurations'].shape}")
    print(f"Compressed NPZ size: {dataset_size_mib:.3f} MiB")
    print(f"Generation time: {main_dataset['elapsed_seconds']:.3f} seconds")
    print(main_validation)
    display(main_saved["manifest"].head(10))
    display(counts)
    display(day23_comparison)
    display(spacing_by_temperature)
    print("Saved files:")
    for name, path in main_saved["paths"].items():
        print(f"  {name}: {path.relative_to(PROJECT_ROOT)}")
else:
    print("Full run skipped because RUN_FULL_DATASET=False.")

chain 001/108: T=1.500000000, replicate=0, start=up, split=train, seed=25000
chain 002/108: T=1.500000000, replicate=1, start=down, split=train, seed=25018
chain 003/108: T=1.500000000, replicate=2, start=up, split=validation, seed=25036
chain 004/108: T=1.500000000, replicate=3, start=down, split=validation, seed=25054
chain 005/108: T=1.500000000, replicate=4, start=up, split=test, seed=25072
chain 006/108: T=1.500000000, replicate=5, start=down, split=test, seed=25090
chain 007/108: T=1.700000000, replicate=0, start=up, split=train, seed=25001
chain 008/108: T=1.700000000, replicate=1, start=down, split=train, seed=25019
chain 009/108: T=1.700000000, replicate=2, start=up, split=validation, seed=25037
chain 010/108: T=1.700000000, replicate=3, start=down, split=validation, seed=25055
chain 011/108: T=1.700000000, replicate=4, start=up, split=test, seed=25073
chain 012/108: T=1.700000000, replicate=5, start=down, split=test, seed=25091
chain 013/108: T=1.900000000, replicate=0, start

,row_index,sample_id,temperature,temperature_index,replicate_index,chain_index,chain_id,seed,initial_state,production_sweep_number,sample_index_within_chain,split_name,energy_per_spin,signed_magnetisation,absolute_magnetisation,acceptance_fraction
0,0,d25_t00_r00_s000,1.5,0,0,0,ising2d_t00_r00_seed25000,25000,up,40,0,train,-1.92,0.980,0.980,0.014250
1,1,d25_t00_r00_s001,1.5,0,0,0,ising2d_t00_r00_seed25000,25000,up,80,1,train,-1.96,0.990,0.990,0.015750
2,2,d25_t00_r00_s002,1.5,0,0,0,ising2d_t00_r00_seed25000,25000,up,120,2,train,-1.85,0.960,0.960,0.014500
3,3,d25_t00_r00_s003,1.5,0,0,0,ising2d_t00_r00_seed25000,25000,up,160,3,train,-1.93,0.980,0.980,0.014500
4,4,d25_t00_r00_s004,1.5,0,0,0,ising2d_t00_r00_seed25000,25000,up,200,4,train,-1.96,0.990,0.990,0.015000
5,5,d25_t00_r00_s005,1.5,0,0,0,ising2d_t00_r00_seed25000,25000,up,240,5,train,-1.93,0.980,0.980,0.012500
6,6,d25_t00_r00_s006,1.5,0,0,0,ising2d_t00_r00_seed25000,25000,up,280,6,train,-1.98,0.995,0.995,0.012563
7,7,d25_t00_r00_s007,1.5,0,0,0,ising2d_t00_r00_seed25000,25000,up,320,7,train,-1.98,0.995,0.995,0.013250
8,8,d25_t00_r00_s008,1.5,0,0,0,ising2d_t00_r00_seed25000,25000,up,360,8,train,-1.94,0.985,0.985,0.014625
9,9,d25_t00_r00_s009,1.5,0,0,0,ising2d_t00_r00_seed25000,25000,up,400,9,train,-1.94,0.980,0.980,0.011563


,temperature,split_name,configuration_count
0,1.500000,test,100
1,1.500000,train,100
2,1.500000,validation,100
3,1.700000,test,100
4,1.700000,train,100
5,1.700000,validation,100
6,1.900000,test,100
7,1.900000,train,100
8,1.900000,validation,100
9,2.050000,test,100


,temperature,day25_energy_per_spin,day25_absolute_magnetisation,day23_energy_per_spin,day23_absolute_magnetisation,energy_difference_day25_minus_day23,absolute_magnetisation_difference_day25_minus_day23
0,1.500000,-1.947900,0.985450,-1.950578,0.986344,0.002678,-0.000894
1,1.700000,-1.899467,0.970533,-1.897718,0.969898,-0.001749,0.000635
2,1.900000,-1.803233,0.935650,-1.811356,0.938763,0.008123,-0.003113
3,2.050000,-1.690767,0.866433,-1.713280,0.896838,0.022513,-0.030404
4,2.100000,-1.653933,0.865417,-1.663516,0.870243,0.009583,-0.004826
5,2.150000,-1.610800,0.839533,-1.614625,0.845798,0.003825,-0.006265
6,2.200000,-1.548733,0.790733,-1.545094,0.785116,-0.003639,0.005617
7,2.230000,-1.512667,0.770450,-1.511786,0.769779,-0.000881,0.000671
8,2.250000,-1.474867,0.727133,-1.459484,0.694281,-0.015383,0.032852
9,2.269185,-1.437133,0.681183,-1.452277,0.706504,0.015144,-0.025320


,temperature,mean_consecutive_hamming_fraction,mean_consecutive_duplicate_rate,mean_signed_magnetisation_lag1
0,1.500000,0.014345,0.006803,0.110446
1,1.700000,0.028980,0.000000,-0.071570
2,1.900000,0.061786,0.000000,-0.082177
3,2.050000,0.112117,0.000000,0.104262
4,2.100000,0.125536,0.000000,-0.033990
5,2.150000,0.146420,0.000000,0.041780
6,2.200000,0.181522,0.000000,0.247203
7,2.230000,0.198350,0.000000,0.569545
8,2.250000,0.225060,0.000000,0.311303
9,2.269185,0.252670,0.000000,0.608590


Saved files:
  dataset_npz: data\raw\classical\ising_ml_dataset_v1.npz
  manifest_csv: data\processed\classical\ising_ml_dataset_v1_manifest.csv
  summary_csv: data\processed\classical\ising_ml_dataset_v1_summary.csv
  chain_diagnostics_csv: data\processed\classical\ising_ml_dataset_v1_chain_diagnostics.csv
  figure_dir: figures\exploratory\classical\day25\main


## References

1. Metropolis, N., Rosenbluth, A. W., Rosenbluth, M. N., Teller, A. H. and Teller, E. (1953), ‘Equation of State Calculations by Fast Computing Machines’, *Journal of Chemical Physics* **21**, 1087–1092. Acceptance procedure: p. 1088.
2. Krauth, W., supplied Section 5.2.1, printed pp. 250–252. Local Ising update: p. 250; slow sampling near the transition: p. 251; ordered and disordered configurations: Fig. 5.17, p. 252.
3. Carrasquilla, J. and Melko, R. G. (2017), ‘Machine learning phases of matter’, *Nature Physics* **13**, 431–434. Raw Ising configurations and separate test configurations: pp. 431–432. No claim from the unavailable supplementary information is used.
4. Onsager, L. (1944), ‘Crystal Statistics. I. A Two-Dimensional Model with an Order-Disorder Transition’, *Physical Review* **65**, 117–149. Infinite-crystal transition condition: p. 117; finite-crystal caution: p. 141.